# NB08: Phenotype Impact Analysis

Compare growth phenotype predictions under Marvin vs OPAM2 thermodynamic
constraints. Test carbon source utilization on minimal media and classify
predictions as CP/CN/FP/FN relative to known E. coli phenotypes.

This evaluates whether OPAM2-derived thermodynamics improve or degrade
the predictive accuracy of metabolic models.

In [1]:
import json
from pathlib import Path

import cobra
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURES_DIR = PROJECT_ROOT / 'figures'

## 1. Load models and deltaG

In [2]:
models = {}
for name in ['ecoli', 'bsubtilis']:
    models[name] = cobra.io.load_json_model(str(DATA_DIR / f'model_{name}.json'))
    sol = models[name].optimize()
    print(f'{name}: {len(models[name].reactions)} reactions, growth={sol.objective_value:.4f}')

with open(DATA_DIR / 'dg_predictions_marvin.json') as f:
    marvin_dg = json.load(f)
with open(DATA_DIR / 'dg_predictions_opam2.json') as f:
    opam2_dg = json.load(f)

DG_THRESHOLD = 5.0

def apply_thermo(model, dg_preds, threshold=DG_THRESHOLD):
    for rxn in model.reactions:
        rxn_id = rxn.id.replace('_c0', '')
        if rxn_id in dg_preds:
            dg = dg_preds[rxn_id]['dG_mean']
            if dg > threshold:
                rxn.upper_bound = min(rxn.upper_bound, 0)
            elif dg < -threshold:
                rxn.lower_bound = max(rxn.lower_bound, 0)

ecoli: 1765 reactions, growth=0.5625


bsubtilis: 1345 reactions, growth=0.5000


## 2. Define phenotype conditions

Carbon source utilization tests based on known E. coli K-12 MG1655 phenotypes.
B. subtilis expected phenotypes are approximate.

In [3]:
carbon_sources = {
    'cpd00027': ('D-Glucose', True),
    'cpd00082': ('D-Fructose', True),
    'cpd00036': ('Succinate', True),
    'cpd00029': ('Acetate', True),
    'cpd00100': ('Glycerol', True),
    'cpd00179': ('Maltose', True),
    'cpd00064': ('Citrate', True),
    'cpd00020': ('Pyruvate', True),
    'cpd00141': ('D-Mannitol', True),
    'cpd00076': ('Sucrose', False),
    'cpd00047': ('Formate', False),
    'cpd00072': ('D-Arabinose', False),
}

essential_exchanges = [
    'EX_cpd00013_e0',  # NH3
    'EX_cpd00009_e0',  # PO4
    'EX_cpd00048_e0',  # SO4
    'EX_cpd00007_e0',  # O2
    'EX_cpd00001_e0',  # H2O
    'EX_cpd00067_e0',  # H+
    'EX_cpd10516_e0',  # Fe3+
    'EX_cpd00034_e0',  # Zn2+
    'EX_cpd00058_e0',  # Cu2+
    'EX_cpd00030_e0',  # Mn2+
    'EX_cpd00254_e0',  # Mg
    'EX_cpd00063_e0',  # Ca2+
    'EX_cpd00149_e0',  # Co2+
    'EX_cpd10515_e0',  # Fe2+
    'EX_cpd00099_e0',  # Cl-
    'EX_cpd00205_e0',  # K+
]

print(f'{len(carbon_sources)} carbon source conditions defined')

12 carbon source conditions defined


## 3. Simulate phenotypes: unconstrained, Marvin, and OPAM2

In [4]:
def simulate_carbon_sources(model, carbon_sources, essential_exchanges):
    """Simulate growth on each carbon source in minimal media."""
    results = []
    for cpd_id, (cpd_name, expected) in carbon_sources.items():
        mdl = model.copy()
        
        # Close all exchanges
        for rxn in mdl.reactions:
            if rxn.id.startswith('EX_'):
                rxn.lower_bound = 0
        
        # Open essentials
        for ex_id in essential_exchanges:
            if ex_id in mdl.reactions:
                mdl.reactions.get_by_id(ex_id).lower_bound = -100
        
        # Open test carbon source
        ex_id = f'EX_{cpd_id}_e0'
        if ex_id not in mdl.reactions:
            results.append((cpd_id, cpd_name, expected, 0.0, False, 'no_exchange'))
            continue
        
        mdl.reactions.get_by_id(ex_id).lower_bound = -10
        sol = mdl.optimize()
        growth = sol.objective_value if sol.status == 'optimal' else 0.0
        predicted = growth > 1e-6
        results.append((cpd_id, cpd_name, expected, growth, predicted, 'ok'))
    
    return results

all_results = []

for org_id, model in models.items():
    print(f'\n=== {org_id} ===')
    
    for label, dg_preds in [('unconstrained', None), ('marvin', marvin_dg), ('opam2', opam2_dg)]:
        mdl = model.copy()
        if dg_preds:
            apply_thermo(mdl, dg_preds)
        
        sim = simulate_carbon_sources(mdl, carbon_sources, essential_exchanges)
        
        for cpd_id, cpd_name, expected, growth, predicted, note in sim:
            if predicted and expected:
                cls = 'CP'
            elif not predicted and not expected:
                cls = 'CN'
            elif predicted and not expected:
                cls = 'FP'
            else:
                cls = 'FN'
            
            all_results.append({
                'organism': org_id,
                'thermo_source': label,
                'cpd_id': cpd_id,
                'cpd_name': cpd_name,
                'expected': expected,
                'growth': round(growth, 6),
                'predicted': predicted,
                'classification': cls,
            })
        
        counts = pd.Series([r[-3] for r in sim]).value_counts()
        n_correct = sum(1 for _, _, exp, _, pred, _ in sim if pred == exp)
        print(f'  {label}: {n_correct}/{len(sim)} correct ({100*n_correct/len(sim):.0f}%)')

df_pheno = pd.DataFrame(all_results)
print(f'\nTotal tests: {len(df_pheno)}')


=== ecoli ===


  unconstrained: 10/12 correct (83%)


  marvin: 3/12 correct (25%)


  opam2: 3/12 correct (25%)

=== bsubtilis ===


  unconstrained: 9/12 correct (75%)


  marvin: 3/12 correct (25%)


  opam2: 3/12 correct (25%)

Total tests: 72


## 4. Accuracy comparison

In [5]:
accuracy_records = []

for org_id in models:
    for label in ['unconstrained', 'marvin', 'opam2']:
        sub = df_pheno[(df_pheno['organism']==org_id) & (df_pheno['thermo_source']==label)]
        cp = (sub['classification']=='CP').sum()
        cn = (sub['classification']=='CN').sum()
        fp = (sub['classification']=='FP').sum()
        fn = (sub['classification']=='FN').sum()
        total = len(sub)
        accuracy = (cp+cn)/total if total > 0 else 0
        
        accuracy_records.append({
            'organism': org_id,
            'thermo': label,
            'CP': cp, 'CN': cn, 'FP': fp, 'FN': fn,
            'accuracy': round(accuracy, 3),
        })
        print(f'{org_id}/{label}: CP={cp} CN={cn} FP={fp} FN={fn} accuracy={accuracy:.1%}')

df_acc = pd.DataFrame(accuracy_records)
print('\n', df_acc.to_string(index=False))

ecoli/unconstrained: CP=8 CN=2 FP=1 FN=1 accuracy=83.3%
ecoli/marvin: CP=0 CN=3 FP=0 FN=9 accuracy=25.0%
ecoli/opam2: CP=0 CN=3 FP=0 FN=9 accuracy=25.0%
bsubtilis/unconstrained: CP=7 CN=2 FP=1 FN=2 accuracy=75.0%
bsubtilis/marvin: CP=0 CN=3 FP=0 FN=9 accuracy=25.0%
bsubtilis/opam2: CP=0 CN=3 FP=0 FN=9 accuracy=25.0%

  organism        thermo  CP  CN  FP  FN  accuracy
    ecoli unconstrained   8   2   1   1     0.833
    ecoli        marvin   0   3   0   9     0.250
    ecoli         opam2   0   3   0   9     0.250
bsubtilis unconstrained   7   2   1   2     0.750
bsubtilis        marvin   0   3   0   9     0.250
bsubtilis         opam2   0   3   0   9     0.250


## 5. Identify flipped predictions

In [6]:
print('Predictions that flipped between Marvin and OPAM2:\n')

for org_id in models:
    marvin_sub = df_pheno[(df_pheno['organism']==org_id) & (df_pheno['thermo_source']=='marvin')]
    opam2_sub = df_pheno[(df_pheno['organism']==org_id) & (df_pheno['thermo_source']=='opam2')]
    
    merged = marvin_sub.merge(opam2_sub, on=['organism', 'cpd_id', 'cpd_name', 'expected'],
                              suffixes=('_marvin', '_opam2'))
    flipped = merged[merged['classification_marvin'] != merged['classification_opam2']]
    
    if len(flipped) > 0:
        print(f'{org_id}: {len(flipped)} flipped predictions')
        for _, row in flipped.iterrows():
            print(f'  {row["cpd_name"]}: {row["classification_marvin"]}→{row["classification_opam2"]} '
                  f'(growth: {row["growth_marvin"]:.4f}→{row["growth_opam2"]:.4f})')
    else:
        print(f'{org_id}: no flipped predictions')

Predictions that flipped between Marvin and OPAM2:

ecoli: no flipped predictions
bsubtilis: no flipped predictions


## 6. Visualize

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, org_id in enumerate(models.keys()):
    ax = axes[idx]
    sub = df_acc[df_acc['organism']==org_id]
    x = np.arange(len(sub))
    colors_map = {'CP': '#4CAF50', 'CN': '#2196F3', 'FP': '#FF9800', 'FN': '#F44336'}
    bottom = np.zeros(len(sub))
    
    for cls in ['CP', 'CN', 'FP', 'FN']:
        vals = sub[cls].values
        ax.bar(x, vals, bottom=bottom, label=cls, color=colors_map[cls])
        bottom += vals
    
    ax.set_xticks(x)
    ax.set_xticklabels(sub['thermo'].values)
    ax.set_ylabel('Count')
    ax.set_title(f'{org_id} — accuracy: {" / ".join(f"{a:.0%}" for a in sub["accuracy"])}')
    ax.legend(loc='upper right')

plt.suptitle('Phenotype Prediction: Unconstrained vs Marvin vs OPAM2')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'phenotype_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved figure to figures/phenotype_comparison.png')

Saved figure to figures/phenotype_comparison.png


## 7. Save results

In [8]:
df_pheno.to_csv(DATA_DIR / 'phenotype_results.tsv', sep='\t', index=False)
df_acc.to_csv(DATA_DIR / 'phenotype_accuracy.tsv', sep='\t', index=False)
print(f'Saved {len(df_pheno)} phenotype results and accuracy summary')

Saved 72 phenotype results and accuracy summary
